# Notebook 00 — Baseline Evaluation
**Assignment 04 | Track 2, Option D | NLP with Deep Learning — IBA**

**What this notebook does:**
1. Defines 10 reasoning prompts (5 general logic + 5 math)
2. Gold answers pre-filled from ChatGPT
3. Runs both `Qwen3-0.6B` and `Qwen3-1.7B` base models **locally** (GPU if available, else CPU)
4. Scores all responses using **Kimi-K2** as LLM-as-Judge via the HF OpenAI-compatible router
5. Saves results to `baseline_results.csv`, `prompts.json`, `gold_answers.json`

**Run this on:** Any machine — uses local inference for base models, HF router API for judge only

In [8]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install -q huggingface_hub pandas transformers torch accelerate python-dotenv openai

In [9]:
import os, json, time, re, pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# ── CONFIG — fill in your HF token ───────────────────────────────────────────
HF_TOKEN = os.environ["HF_TOKEN"]

BASE_MODELS = [
    "Qwen/Qwen3-0.6B",
    "Qwen/Qwen3-1.7B",
]

# Judge must differ from gold-answer generator (assignment rule)
# Gold answers: generated via ChatGPT (pasted below)
# Judge: Kimi via HF OpenAI-compatible router (different from ChatGPT gold generator)
# Mistral-7B-Instruct-v0.3 currently fails on HF provider routing for this account.
JUDGE_MODEL = "moonshotai/Kimi-K2-Instruct:novita"
JUDGE_CLIENT = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=HF_TOKEN,
)

# Alternative: if 3090 Ti has Ollama set up with Jackrong's 27B model:
# Use ollama python client instead — swap judge calls at the bottom

## Step 1 — Define 10 Reasoning Prompts
5 general logic + 5 math word problems.
These same 10 prompts will be used throughout ALL notebooks for evaluation.

In [10]:
PROMPTS = [
    # ── General Logic (5) ────────────────────────────────────────────────────
    {
        "id": "G1",
        "type": "general",
        "prompt": (
            "Alice, Bob, and Carol each own exactly one pet: a cat, a dog, or a fish. "
            "Alice does not own the cat. Bob does not own the dog. "
            "Carol does not own the fish. Who owns which pet? Reason step by step."
        )
    },
    {
        "id": "G2",
        "type": "general",
        "prompt": (
            "A bat and a ball together cost $1.10. The bat costs $1.00 more than the ball. "
            "How much does the ball cost? Show your reasoning carefully."
        )
    },
    {
        "id": "G3",
        "type": "general",
        "prompt": (
            "In a room of 30 people, is it more likely than not that at least two share a birthday? "
            "Explain your reasoning step by step without necessarily computing the exact probability."
        )
    },
    {
        "id": "G4",
        "type": "general",
        "prompt": (
            "If all Bloops are Razzies, and all Razzies are Lazzies, are all Bloops definitely Lazzies? "
            "What logical principle applies here? Explain step by step."
        )
    },
    {
        "id": "G5",
        "type": "general",
        "prompt": (
            "A farmer has 17 sheep. All but 9 die. How many sheep are left? "
            "Explain your reasoning before giving the answer."
        )
    },
    # ── Math Word Problems (5) ────────────────────────────────────────────────
    {
        "id": "M1",
        "type": "math",
        "prompt": (
            "A train travels at 60 km/h for 2 hours, then at 90 km/h for 1.5 hours. "
            "What is the total distance travelled? Show all working."
        )
    },
    {
        "id": "M2",
        "type": "math",
        "prompt": (
            "Find the sum of all integers from 1 to 100. "
            "Show the formula you use and explain why it works."
        )
    },
    {
        "id": "M3",
        "type": "math",
        "prompt": (
            "A store sells an item at a 20% discount. After the discount the price is $80. "
            "What was the original price? Show step-by-step working."
        )
    },
    {
        "id": "M4",
        "type": "math",
        "prompt": (
            "Two pipes can fill a tank in 6 hours and 4 hours respectively. "
            "If both pipes are opened simultaneously, how long will it take to fill the tank? "
            "Show all reasoning."
        )
    },
    {
        "id": "M5",
        "type": "math",
        "prompt": (
            "A sequence starts: 2, 6, 18, 54, ... "
            "What is the 8th term? Identify the pattern and show your working."
        )
    },
]

print(f"Defined {len(PROMPTS)} prompts: {sum(1 for p in PROMPTS if p['type']=='general')} general, {sum(1 for p in PROMPTS if p['type']=='math')} math")

Defined 10 prompts: 5 general, 5 math


## Step 2 — Gold Answers (generated via ChatGPT)

**Action required:** Copy each prompt above into ChatGPT, get its response, paste below.
These serve as the reference answers for LLM-as-Judge scoring.

Fill in the `gold_answers` dict below — keys match prompt IDs.

In [11]:
gold_answers = {
    "G1": """We'll solve it step by step.

We have:
- Pets: cat, dog, fish
- People: Alice, Bob, Carol
- Each person owns exactly one different pet.

Clues:
1. Alice does not own the cat.
2. Bob does not own the dog.
3. Carol does not own the fish.

Step 1: Suppose Alice owns the dog.
Then the remaining pets are cat and fish for Bob and Carol.
- Bob cannot own the dog (already true).
- Carol cannot own the fish.
So Carol must own the cat, and Bob must own the fish.
This works: Alice → dog, Bob → fish, Carol → cat.

Step 2: Check if another solution exists.
Suppose Alice owns the fish instead.
Then Bob and Carol must have cat and dog.
- Bob cannot own the dog → Bob must own the cat.
- Carol then owns the dog.
This also satisfies all clues: Alice → fish, Bob → cat, Carol → dog.

So the information does not give a unique answer. The valid solutions are:
1. Alice → dog, Bob → fish, Carol → cat
2. Alice → fish, Bob → cat, Carol → dog""",

    "G2": """Let the ball cost x dollars.
Then the bat costs x + 1.00 (because the bat costs $1 more than the ball).

Add them together:
x + (x + 1.00) = 1.10
2x + 1.00 = 1.10
2x = 0.10
x = 0.05

So the ball costs $0.05.
And the bat costs $1.05, which together make $1.10.""",

    "G3": """Yes — in a room of 30 people, it is more likely than not that at least two people share a birthday. This is the famous "birthday paradox."

Step 1: Think about the opposite event.
Instead of asking "What is the chance that two people share a birthday?", ask "What is the chance that everyone has different birthdays?" If the probability that everyone is different drops below 50%, then the probability of at least one match must be above 50%.

Step 2: Build the probability person by person.
Assume 365 equally likely birthdays and ignore leap years.
- The first person can have any birthday: 365/365 = 1
- The second person must avoid the first person's birthday: 364/365
- The third person must avoid the first two birthdays: 363/365
- And so on.
So the probability that all 30 birthdays are different is: (365/365) × (364/365) × (363/365) × ... × (336/365)

Step 3: Notice what happens.
With 30 people, there are (30 × 29)/2 = 435 different pairs of people. That is a lot of opportunities for a match.

Step 4: Intuition.
A common mistake is thinking "30 is far less than 365, so matching birthdays should be unlikely." But the important thing is not comparing people to days — it is comparing the huge number of possible pairs. By 23 people, the probability of a shared birthday is already above 50%. So with 30 people, it is definitely more likely than not.

Answer: Yes, with 30 people it is more likely than not that two share a birthday.""",

    "G4": """Yes — all Bloops are definitely Lazzies.

Step 1: Write the statements clearly.
We are told:
1. All Bloops are Razzies.
2. All Razzies are Lazzies.

In set language:
- Bloops ⊆ Razzies
- Razzies ⊆ Lazzies

Step 2: Follow the chain.
Take any Bloop.
- Since all Bloops are Razzies, that Bloop must be a Razzie.
- Since all Razzies are Lazzies, that Razzie must also be a Lazzie.
Therefore: every Bloop is a Lazzie.

Step 3: The logical principle.
This uses the transitive property (or transitive reasoning) of class inclusion.
General form: If all A are B, and all B are C, then all A are C. Symbolically: A ⊆ B, B ⊆ C → A ⊆ C.

Step 4: Simple analogy.
- All cats are mammals.
- All mammals are animals.
- Therefore, all cats are animals.
Same logical structure.""",

    "G5": """Let's read the sentence carefully: "A farmer has 17 sheep. All but 9 die."

The key phrase is "All but 9." This means every sheep dies except 9 sheep. So 9 sheep survive.

Another way to think about it:
- Start with 17 sheep.
- "All but 9 die" = all the others die.
- Therefore, the number remaining alive is 9.

The answer is 9.""",

    "M1": """Use the basic formula: Distance = Speed × Time

Step 1: First part of the journey.
Speed = 60 km/h, Time = 2 hours
Distance = 60 × 2 = 120 km

Step 2: Second part of the journey.
Speed = 90 km/h, Time = 1.5 hours
Distance = 90 × 1.5 = 135 km

Step 3: Add the distances.
120 + 135 = 255 km

The total distance travelled is 255 km.""",

    "M2": """We want to find: 1 + 2 + 3 + ... + 100

Use the formula for the sum of the first n integers: n(n+1)/2
Here, n = 100.

So: 100 × 101 / 2 = 50 × 101 = 5050

The sum is 5050.

Why the formula works:
Write the numbers forward and backward:
1 + 2 + 3 + ... + 98 + 99 + 100
100 + 99 + 98 + ... + 3 + 2 + 1

Add vertically: each pair equals 101. There are 100 such pairs, so 100 × 101 = 10100.
But this counts the sum twice, so divide by 2: 10100 / 2 = 5050.
That gives the general formula: n(n+1)/2.""",

    "M3": """Let the original price be x.
A 20% discount means the customer pays 100% - 20% = 80% of the original price.
We are told the discounted price is $80, so:
0.8x = 80
x = 80 / 0.8
x = 100

The original price was $100.

Quick check: 20% of $100 is 0.20 × 100 = $20. Subtract the discount: 100 - 20 = $80. Correct.""",

    "M4": """Step 1: Find each pipe's filling rate.
Pipe A fills the tank in 6 hours → fills 1/6 of the tank per hour.
Pipe B fills the tank in 4 hours → fills 1/4 of the tank per hour.

Step 2: Add the rates together.
1/6 + 1/4 = 2/12 + 3/12 = 5/12
Together, the pipes fill 5/12 of the tank per hour.

Step 3: Find the total time.
Time = 1 / (5/12) = 12/5 = 2.4 hours.

Step 4: Convert to hours and minutes.
0.4 hour = 0.4 × 60 = 24 minutes.

The total time is 2 hours 24 minutes.""",

    "M5": """The sequence is: 2, 6, 18, 54, ...

Step 1: Identify the pattern.
2 × 3 = 6, 6 × 3 = 18, 18 × 3 = 54.
Each term is multiplied by 3. This is a geometric sequence with first term a = 2 and common ratio r = 3.

Step 2: Use the geometric sequence formula.
The nth term is: a_n = a × r^(n-1)
For the 8th term: a_8 = 2 × 3^7 (because 8-1 = 7)

Step 3: Calculate.
3^7 = 2187
2 × 2187 = 4374

The 8th term is 4374.""",
}

assert all(v != "PASTE CHATGPT ANSWER HERE" for v in gold_answers.values()), \
    "Fill in all gold answers before proceeding!"
print("Gold answers loaded.")

Gold answers loaded.


## Step 3 — Run Base Models Locally

Qwen3-0.6B and Qwen3-1.7B are **not available on the HF Inference API free tier**.
Both models run locally via `transformers` on CPU (float16, ~1.2GB and ~3.4GB RAM respectively).
Each model is loaded once, all 10 prompts run, then the model is unloaded before the next one.

In [12]:
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype  = torch.float16 if device == "cuda" else torch.float32
print(f"Using device: {device} | dtype: {dtype}")

def _apply_chat_template(tokenizer, messages):
    """apply_chat_template returns a plain tensor in older transformers and
    a BatchEncoding dict in newer ones (4.50+). This handles both."""
    encoded = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    )
    return encoded["input_ids"] if not isinstance(encoded, torch.Tensor) else encoded

def run_model_local(model_id, prompts, hf_token, max_new_tokens=512):
    """Load model locally, run all prompts, unload. Uses GPU if available, else CPU."""
    print(f"  Loading {model_id} on {device}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        token=hf_token,
        torch_dtype=dtype,
        device_map="auto" if device == "cuda" else "cpu",
    )
    model.eval()

    responses = {}
    for p in prompts:
        print(f"  Prompt {p['id']}...", end=" ", flush=True)
        messages = [
            {"role": "system", "content": "You are a helpful reasoning assistant. Think step by step."},
            {"role": "user",   "content": p["prompt"]}
        ]
        input_ids = _apply_chat_template(tokenizer, messages).to(device)
        with torch.no_grad():
            output = model.generate(
                input_ids,
                max_new_tokens=max_new_tokens,
                temperature=0.1,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        response = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
        responses[p["id"]] = response
        print("done")

    del model, tokenizer
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()
    return responses


# Run both base models on all 10 prompts
base_responses = {}

for model_id in BASE_MODELS:
    print(f"\n── Running {model_id} ──")
    base_responses[model_id] = run_model_local(model_id, PROMPTS, HF_TOKEN)

print("\nAll base model responses collected.")

Using device: cuda | dtype: torch.float16

── Running Qwen/Qwen3-0.6B ──
  Loading Qwen/Qwen3-0.6B on cuda...


Loading weights: 100%|██████████| 311/311 [00:00<00:00, 872.55it/s]


  Prompt G1... done
  Prompt G2... done
  Prompt G3... done
  Prompt G4... done
  Prompt G5... done
  Prompt M1... done
  Prompt M2... done
  Prompt M3... done
  Prompt M4... done
  Prompt M5... done

── Running Qwen/Qwen3-1.7B ──
  Loading Qwen/Qwen3-1.7B on cuda...


Loading weights: 100%|██████████| 311/311 [00:00<00:00, 562.08it/s]


  Prompt G1... done
  Prompt G2... done
  Prompt G3... done
  Prompt G4... done
  Prompt G5... done
  Prompt M1... done
  Prompt M2... done
  Prompt M3... done
  Prompt M4... done
  Prompt M5... done

All base model responses collected.


## Step 4 — LLM-as-Judge Scoring via HF OpenAI-Compatible Router

Judge: `moonshotai/Kimi-K2-Instruct:novita` (different from ChatGPT gold generator ✓)

Scoring: 1–5 scale on reasoning quality
- **5**: Correct answer, clear step-by-step reasoning, no errors
- **4**: Correct answer, reasoning mostly clear with minor gaps
- **3**: Partially correct or reasoning has gaps
- **2**: Wrong answer but shows some reasoning attempt
- **1**: Wrong answer, no meaningful reasoning

In [13]:
JUDGE_PROMPT_TEMPLATE = """\
You are an expert evaluator of reasoning quality.

QUESTION:
{question}

REFERENCE ANSWER (Gold):
{gold}

CANDIDATE RESPONSE:
{response}

Evaluate the candidate response on REASONING QUALITY using this scale:
5 = Correct answer, clear step-by-step reasoning, no errors
4 = Correct answer, reasoning mostly clear with minor gaps
3 = Partially correct or reasoning has notable gaps
2 = Wrong answer but shows some reasoning attempt
1 = Wrong answer, no meaningful reasoning

Respond in this exact format (nothing else):
SCORE: <number 1-5>
REASON: <one sentence justification>"""


def _parse_judge_output(out):
    score_match = re.search(r"SCORE\s*:\s*([1-5])", out, re.IGNORECASE)
    reason_match = re.search(r"REASON\s*:\s*(.+)", out, re.IGNORECASE | re.DOTALL)
    if not score_match:
        raise ValueError(f"Judge response did not include SCORE 1-5. Raw output: {out!r}")
    reason = reason_match.group(1).strip() if reason_match else out.strip()
    return int(score_match.group(1)), reason


def judge_response(question, gold, response):
    """Score a response using Kimi through the HF OpenAI-compatible router."""
    prompt = JUDGE_PROMPT_TEMPLATE.format(
        question=question, gold=gold, response=response
    )
    last_error = None
    for attempt in range(3):
        try:
            completion = JUDGE_CLIENT.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=180,
                temperature=0.0,
            )
            out = completion.choices[0].message.content.strip()
            return _parse_judge_output(out)
        except Exception as error:
            last_error = error
            time.sleep(2 * (attempt + 1))
    return 0, f"ERROR after retries: {last_error}"


print(f"Judge helper version: hf-router-openai | model={JUDGE_MODEL}")


# If the kernel was restarted, reuse already-generated responses from the previous CSV.
if "base_responses" not in globals() and os.path.exists("baseline_results.csv"):
    old_df = pd.read_csv("baseline_results.csv")
    if {"model", "prompt_id", "response"}.issubset(old_df.columns):
        base_responses = {
            model_id: dict(zip(group["prompt_id"], group["response"]))
            for model_id, group in old_df.groupby("model")
        }
        print("Loaded existing model responses from baseline_results.csv; rescoring only.")
if "base_responses" not in globals():
    raise RuntimeError("No saved/generated base_responses found. Run Step 3 once to generate Qwen responses, then rerun Step 4/5.")

# Score all responses
records = []
for model_id in BASE_MODELS:
    print(f"\n── Judging {model_id} ──")
    for p in PROMPTS:
        print(f"  Prompt {p['id']}...", end=" ")
        response = base_responses[model_id][p["id"]]
        score, reason = judge_response(p["prompt"], gold_answers[p["id"]], response)
        records.append({
            "model":    model_id,
            "stage":    "baseline",
            "trial":    "base",
            "prompt_id": p["id"],
            "type":     p["type"],
            "response": response,
            "score":    score,
            "reason":   reason,
        })
        print(f"score={score}")
        time.sleep(1)

print("\nJudging complete.")

Judge helper version: chat-first v3 | model=moonshotai/Kimi-K2-Instruct

── Judging Qwen/Qwen3-0.6B ──
  Prompt G1... score=3
  Prompt G2... score=5
  Prompt G3... score=3
  Prompt G4... score=4
  Prompt G5... score=2
  Prompt M1... score=5
  Prompt M2... score=5
  Prompt M3... score=5
  Prompt M4... score=5
  Prompt M5... score=5

── Judging Qwen/Qwen3-1.7B ──
  Prompt G1... score=4
  Prompt G2... score=5
  Prompt G3... score=4
  Prompt G4... score=5
  Prompt G5... score=5
  Prompt M1... score=5
  Prompt M2... score=5
  Prompt M3... score=5
  Prompt M4... score=5
  Prompt M5... score=5

Judging complete.


## Step 5 — Results Summary

In [14]:
df = pd.DataFrame(records)

# Summary table
summary = df.groupby("model")["score"].agg(["mean", "sum", "count"]).round(2)
summary.columns = ["avg_score", "total_score", "n_prompts"]
print("\n── Baseline Summary ──")
print(summary.to_string())

print("\n── Per-prompt scores ──")
pivot = df.pivot_table(index="prompt_id", columns="model", values="score")
print(pivot.to_string())

# Save
df.to_csv("baseline_results.csv", index=False)
with open("gold_answers.json", "w") as f:
    json.dump(gold_answers, f, indent=2)
with open("prompts.json", "w") as f:
    json.dump(PROMPTS, f, indent=2)

print("\nSaved: baseline_results.csv, gold_answers.json, prompts.json")
print("\nThese files are needed by notebooks 01, 02, and 03.")


── Baseline Summary ──
                 avg_score  total_score  n_prompts
model                                             
Qwen/Qwen3-0.6B        4.2           42         10
Qwen/Qwen3-1.7B        4.8           48         10

── Per-prompt scores ──
model      Qwen/Qwen3-0.6B  Qwen/Qwen3-1.7B
prompt_id                                  
G1                     3.0              4.0
G2                     5.0              5.0
G3                     3.0              4.0
G4                     4.0              5.0
G5                     2.0              5.0
M1                     5.0              5.0
M2                     5.0              5.0
M3                     5.0              5.0
M4                     5.0              5.0
M5                     5.0              5.0

Saved: baseline_results.csv, gold_answers.json, prompts.json

These files are needed by notebooks 01, 02, and 03.
